# TA News Recommendation - EDA & Experimentation

Notebook untuk eksplorasi data, preprocessing, dan eksperimen model.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw'
PROC_DIR = DATA_DIR / 'processed'

print('Directories:')
print(f'  Raw: {RAW_DIR.exists()}')
print(f'  Processed: {PROC_DIR.exists()}')

## 1. Load & Inspect Raw Data

In [ ]:
# Load raw news data
raw_files = list(RAW_DIR.glob('*.csv')) + list(RAW_DIR.glob('*.jsonl')) + list(RAW_DIR.glob('*.parquet'))
raw_files

In [ ]:
if raw_files:
    df = pd.read_csv(raw_files[0]) if raw_files[0].suffix == '.csv' else pd.read_parquet(raw_files[0])
    print(f'Shape: {df.shape}')
    print(f'Columns: {df.columns.tolist()}')
    print(f'Dtypes:\n{df.dtypes}')
    display(df.head())
    print(f'\nMissing values:\n{df.isnull().sum()}')
    print(f'\nDate range: {df["publish_date"].min()} to {df["publish_date"].max()}')
else:
    print('No raw data files found. Download dataset first.'
          '\nSuggested: kaggle datasets download -d <indonesian-news-dataset>')

## 2. Category Distribution

In [ ]:
if 'df' in locals() and 'category' in df.columns:
    cat_counts = df['category'].value_counts()
    print(cat_counts)
    
    plt.figure(figsize=(10, 5))
    cat_counts.plot(kind='bar')
    plt.title('Distribusi Kategori Berita')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 3. Text Length Analysis

In [ ]:
if 'df' in locals() and 'content' in df.columns:
    df['content_length'] = df['content'].astype(str).str.len()
    df['word_count'] = df['content'].astype(str).str.split().str.len()
    
    print(df[['content_length', 'word_count']].describe())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    df['content_length'].hist(bins=50, ax=axes[0])
    axes[0].set_title('Character Length Distribution')
    df['word_count'].hist(bins=50, ax=axes[1])
    axes[1].set_title('Word Count Distribution')
    plt.tight_layout()
    plt.show()

## 4. Temporal Analysis

In [ ]:
if 'df' in locals() and 'publish_date' in df.columns:
    df['publish_date'] = pd.to_datetime(df['publish_date'])
    df['year_month'] = df['publish_date'].dt.to_period('M')
    
    monthly = df.groupby('year_month').size()
    monthly.plot(kind='line', marker='o')
    plt.title('Jumlah Berita per Bulan')
    plt.xlabel('Bulan')
    plt.ylabel('Jumlah')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 5. Preprocessing Test

In [ ]:
from preprocessing.run import IndonesianPreprocessor

preprocessor = IndonesianPreprocessor(
    stopwords_path='../configs/stopwords_id.txt',
    min_token_len=2,
    max_token_len=50
)

# Test pada sample
sample_text = "Pemerintah mengumumkan kebijakan baru soal subsidi BBM yang berlaku mulai bulan depan. Masyarakat diharapkan sabar."
print('Original:', sample_text)
print('Processed:', preprocessor.process(sample_text))

## 6. Run Full Preprocessing Pipeline

In [ ]:
!python -m src.preprocessing.run --config ../configs/preprocessing.yaml --input ../data/raw/news.csv --output ../data/processed/news_processed.parquet

## 7. Train TF-IDF Baseline

In [ ]:
!python -m src.models.content_based.train --config ../configs/tfidf_baseline.yaml

## 8. Train ALS Collaborative Filtering

In [ ]:
!python -m src.models.collaborative.train --config ../configs/als.yaml

## 9. Train Weighted Hybrid

In [ ]:
!python -m src.models.hybrid.train --config ../configs/hybrid.yaml

## 10. Evaluate Models

In [ ]:
!python -m src.evaluation.run --model tfidf --config ../configs/tfidf_baseline.yaml
!python -m src.evaluation.run --model als --config ../configs/als.yaml
!python -m src.evaluation.run --model hybrid --config ../configs/hybrid.yaml

## 11. Start API Server

In [ ]:
# Run in terminal, not notebook:
# uvicorn src.api.main:app --reload --port 8000

## 12. Test API

In [ ]:
import requests

# Test health
r = requests.get('http://localhost:8000/health')
print(r.json())

# Test recommend (need trained models and user/item in maps)
# r = requests.post('http://localhost:8000/recommend', json={
#     'user_id': 'user_123',
#     'top_k': 5,
#     'model': 'hybrid'
# })
# print(r.json())